In [ ]:
# Định nghĩa các giá trị mặc định (Papermill sẽ đè lên các giá trị này)
input_surface = "default_path.stl"
E = 0.01
nu = 0.09
cell_size = 8

In [12]:
import json, sys, copy, os, igl
import numpy as np
import meshio

def is_inside(v, f, p):
    winds = igl.winding_number(v, f, p)
    mask = np.array(np.rint(winds), dtype=int)
    
    return np.array([e % 2 == 1 for e in mask])

def change_file_extension(filename, new_extension):
  base, ext = os.path.splitext(filename)
  return base + "." + new_extension if ext else filename + "." + new_extension

In [ ]:
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
microstructure_repo_path = notebook_dir
matopt_repo_path = os.path.join(os.path.dirname(microstructure_repo_path), "matopt")

# input_surface = input_surface = "/mnt/d/3_Aboard/Project/modeling/stl/8cube.STL" #"/mnt/c/Users/Asus/desktop/Part1.STL" # USER: Change this to your STL file path"C:\Users\ASUS\Desktop\Part1.STL"
stitch_cells_cli = os.path.join(microstructure_repo_path, "build/isosurface_inflator/cut_cells_cli")
only_cube_cells = False

In [ ]:
# E = 0.01
# nu = 0.09
# cell_size = 8
# k = 3
# out_path = str(E) + "_" + str(cell_size) + ".obj"
out_path = f"{input_surface[:-4]}_{E}_{cell_size}.obj_{only_cube_cells}.obj"
print(out_path)

m = meshio.read(input_surface)
input_surface = change_file_extension(input_surface, 'obj')

v = m.points.astype(float)

# --- ĐOẠN XỬ LÝ CĂN GIỮA VÀ ÉP LƯỚI ĐỐI XỨNG ---
# 1. Dời trọng tâm của file 3D về chính xác gốc tọa độ (0,0,0)
center = (np.amax(v, axis=0) + np.amin(v, axis=0)) / 2.0
v = v - center
m.points = v
m.write(input_surface) # Ghi đè lại file để phần mềm C++ lõi nhận đúng tọa độ tâm

# 2. Tính toán Bounding Box mới đã được căn giữa
bbox = [np.amin(v, axis=0), np.amax(v, axis=0)]

# 3. Ép buộc số lượng ô lưới phải đối xứng hai bên trục (từ -N đến +N)
max_dims = np.amax(np.abs(bbox), axis=0)
corner1 = list(map(int, np.ceil(max_dims / cell_size)))
corner0 = [-c for c in corner1] 
# -----------------------------------------------------

print("Góc lưới đối xứng:", corner0, corner1)
#f = m.cells.data


# use return E for uniform Young's modulus
# use i,j,k to vary the Young's modulus through the dimensions
def young(i, j, k):
    # return E
    print(k)
    if k == 0:
        return 0.001
    if k == 1:
        return 0.0015
    if k == 2:
        return 0.002
    if k == 3:  
        return 0.005
    if k == 4:
        return 0.005
    if k == 5:
        return 0.005
    if k == 6:
        return 0.005
    if k == 7:
        return 0.005
    if k == 8:
        return 0.005
    if k == 9:
        return 0.005
    return E
    

/mnt/d/3_Aboard/Project/modeling/stl/8cube_0.01_8.obj_False.obj
Góc lưới đối xứng: [-1, -1, -1] [1, 1, 1]


In [ ]:
# Sửa lại đoạn này trong Notebook của bạn
notebook_dir = os.getcwd() 
# Đảm bảo đường dẫn này trỏ đúng vào file thực thi cut_cells_cli của bạn trong WSL
stitch_cells_cli = "/mnt/d/miCADstructure/Tools/microstructure_inflators/build/isosurface_inflator/cut_cells_cli"# Ép file đầu ra vào folder Data_Output để không bị thất lạc
output_folder = "/mnt/d/miCADstructure/Data_Output"
filename_only = os.path.basename(input_surface)[:-4]
out_path = os.path.join(output_folder, f"{filename_only}_{E}_{cell_size}.obj_{only_cube_cells}.obj")

In [ ]:
import sys
import os

# Đường dẫn tuyệt đối trong WSL
matopt_repo_path = "/mnt/d/miCADstructure/Tools/matopt"
module_path = os.path.join(matopt_repo_path, "tools/material2geometry")

# Nạp đường dẫn vào hệ thống để import được
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from material2geometry import Material2Geometry

# Gọi file cấu hình
coeffs_file = os.path.join(module_path, "0646_geo_1_coeffs.txt")
mat2geo = Material2Geometry(in_path=coeffs_file)

Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]


In [17]:
patterns = []
entry = {"params": [],
"symmetry": "Cubic",
"pattern": os.path.join(microstructure_repo_path, "data/patterns/3D/reference_wires/pattern0646.wire"),
"index": [0,0,0]}

for i in range(corner0[0], 1 + corner1[0]):
    for j in range(corner0[1], 1 + corner1[1]):
        for k in range(corner0[2], 1 + corner1[2]):
            geo_params = mat2geo.evaluate(nu, young(i, j, k))
            entry["params"] = geo_params
            entry["index"] = [i,j,k]
            patterns.append(copy.deepcopy(entry))

with open("data.json", 'w') as f:
    json.dump(patterns, f)

-1
0
1
-1
0
1
-1
0
1
-1
0
1
-1
0
1
-1
0
1
-1
0
1
-1
0
1
-1
0
1


In [18]:
os.system(stitch_cells_cli + " -p data.json " +
           (("--surface " + input_surface) if not only_cube_cells else "") + 
           " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50")
# print((stitch_cells_cli + " -p data.json " +
#            (("--surface " + input_surface) if not only_cube_cells else "") + 
#            " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50"))

Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 3.7037 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 7.40741 %
Checking printability
Inflation graph generated
Saved space 71.1232 %
Completed 11.1111 %
Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 14.8148 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 18.5185 %
Checking printability
Inflation graph generated
Saved space 71.1232 %
Completed 22.2222 %
Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 25.9259 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 29.6296 %
Checking printability
Inflation graph generated
Saved space 71.1232 %
Completed 33.3333 %
Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 37.037 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 40.7407 %
Checking printab

0

In [19]:
import subprocess

print('stitch_cells_cli:', stitch_cells_cli)
print('exists:', os.path.exists(stitch_cells_cli))

cmd = [stitch_cells_cli, '-p', 'data.json']
if not only_cube_cells:
    cmd += ['--surface', input_surface]
cmd += ['--gridSize', str(cell_size), '-o', out_path, '-r', '50']

print('running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', res.returncode)
print('stdout:\n', res.stdout)
print('stderr:\n', res.stderr)


stitch_cells_cli: /mnt/d/eFlesh-main/eFlesh-main/microstructure/microstructure_inflators/build/isosurface_inflator/cut_cells_cli
exists: True
running: /mnt/d/eFlesh-main/eFlesh-main/microstructure/microstructure_inflators/build/isosurface_inflator/cut_cells_cli -p data.json --surface /mnt/d/3_Aboard/Project/modeling/stl/8cube.obj --gridSize 8 -o /mnt/d/3_Aboard/Project/modeling/stl/8cube_0.01_8.obj_False.obj -r 50
returncode: 0
stdout:
 Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 3.7037 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 7.40741 %
Checking printability
Inflation graph generated
Saved space 71.1232 %
Completed 11.1111 %
Checking printability
Inflation graph generated
Saved space 65.7664 %
Completed 14.8148 %
Checking printability
Inflation graph generated
Saved space 71.296 %
Completed 18.5185 %
Checking printability
Inflation graph generated
Saved space 71.1232 %
Completed 22.2222 %
Checking printability


In [20]:
for c in [1, 2, 3, 4, 5, 6, 7, 8]:
    for E in [1e-3, 2e-3, 5e-3, 1e-2]:
        print(f"Processing for c={c} and E={E}")
        print((np.array(mat2geo.evaluate(nu, E)[5:-4]) * c).tolist())

Processing for c=1 and E=0.001
[0.12500002236530952, 0.08039353577480896, 0.10304258195535204, 0.15000002683837113]
Processing for c=1 and E=0.002
[0.12500002321509995, 0.08246950095234824, 0.10661763958398165, 0.15000002785811953]
Processing for c=1 and E=0.005
[0.12500002462842463, 0.0886881045353454, 0.11693105096104062, 0.15000002955410885]
Processing for c=1 and E=0.01
[0.12500002371577018, 0.09894629144936973, 0.1327853472775627, 0.15000002845892332]
Processing for c=2 and E=0.001
[0.25000004473061904, 0.1607870715496179, 0.20608516391070408, 0.30000005367674226]
Processing for c=2 and E=0.002
[0.2500000464301999, 0.1649390019046965, 0.2132352791679633, 0.30000005571623906]
Processing for c=2 and E=0.005
[0.25000004925684927, 0.1773762090706908, 0.23386210192208123, 0.3000000591082177]
Processing for c=2 and E=0.01
[0.25000004743154036, 0.19789258289873946, 0.2655706945551254, 0.30000005691784665]
Processing for c=3 and E=0.001
[0.3750000670959286, 0.24118060732442687, 0.30912774

In [21]:
# To check printability of the geometry, we can create a DataFrame to summarize the minimum dimensions for different cell sizes and Young's moduli.Based on your printer nozzle size, we can determine if the geometry is printable or not.

import pandas as pd
import numpy as np

c_values = [1, 2, 3, 4, 5, 6, 7, 8, 16]
E_values = [1e-3, 2e-3, 5e-3, 1e-2, 2e-2]

df = pd.DataFrame(index=c_values, columns=E_values)

df.index.name = "Cell Size (c)"
df.columns.name = "Young's Modulus (E)"

for c in c_values:
    for E in E_values:
        arr = np.array(mat2geo.evaluate(nu, E)[5:-4]) * c
        min_dim = np.min(arr)        
        if min_dim < 0.4:
            df.loc[c, E] = "Unprintable"
        else:
            df.loc[c, E] = f"{min_dim:.4f}"

df

Young's Modulus (E),0.001,0.002,0.005,0.010,0.020
Cell Size (c),,,,,
1,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
2,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
3,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
4,Unprintable,Unprintable,Unprintable,Unprintable,0.4740
5,0.4020,0.4123,0.4434,0.4947,0.5925
6,0.4824,0.4948,0.5321,0.5937,0.7110
7,0.5628,0.5773,0.6208,0.6926,0.8295
8,0.6431,0.6598,0.7095,0.7916,0.9480
16,1.2863,1.3195,1.4190,1.5831,1.8960


In [22]:
import os

input_surface = r"C:\Users\ASUS\Desktop\input.STL"
E = 0.01
cell_size = 8
only_cube_cells = False

out_path = f"{input_surface[:-4]}_{E}_{cell_size}.obj_{only_cube_cells}.obj"
print("Expected output file:", out_path)
print("Exists:", os.path.exists(out_path))

Expected output file: C:\Users\ASUS\Desktop\input_0.01_8.obj_False.obj
Exists: False
